# Isaac Lab과 Robot Simulation


## 학습 목표

<!-- exercise-guidance:reinforcement-learning/week-09/objective -->
새 Simulator의 예제를 실행하는 데서 멈추지 않고, 내 연구 Task에서 바꿔야 할 관측·행동·물리를 찾습니다.
<!-- /exercise-guidance:reinforcement-learning/week-09/objective -->


## 실행 조건

<!-- exercise-guidance:reinforcement-learning/week-09/execution -->
CPU 경로는 Gymnasium·NumPy·Matplotlib을 사용합니다. 공식 SO-101 튜토리얼은 Python 3.12·uv·Git LFS와 Linux/Windows NVIDIA GPU 환경을 요구합니다. GPU 환경 준비는 수업 전 별도 점검합니다. 개인 GPU 실행을 권장하고 공용 장비로 보완합니다. Jetson은 Edge 추론·지연 측정 확장에 사용합니다.
<!-- /exercise-guidance:reinforcement-learning/week-09/execution -->


## 적용 사례

<!-- exercise-guidance:reinforcement-learning/week-09/case-studies -->
### 이동로봇의 경로와 안전

격자 경로 또는 연속 속도 제어 중 한 가지를 선택하는 적용 설계 사례입니다.

**Observation·State:** 위치·목표 상대 위치·근접 장애물 관측.

**행동:** 격자 방향 선택 또는 제한된 Linear Velocity·Angular Velocity.

**Reward·Constraint:** 도착·충돌·이동 비용을 분리. 충돌과 시간 초과를 구분.

**비교 기준:** 최단 경로, 규칙 기반 회피.

**확인할 지표:** 성공률, 충돌률, 이동 거리, 시간. 미관측 지도에서 평가.

### 구동계의 추종 성능과 입력 비용

각 실험에서 일정한 Reference Angular Velocity를 추종하는 합성 1차 Motor 모델입니다. 학습은 목표 1, 평가는 0.5·1·1.5에서 구분합니다.

**Observation·State:** Angular Velocity ω/5, 목표 r/2.

**행동:** 연속 a∈[−1,1], 실제 입력 u=2a.

**Reward·Constraint:** −((r−ω)²+0.05u²). |ω|>5에서 종료, 100 step에서 시간 제한.

**비교 기준:** 동일 입력 제한의 P 제어.

**확인할 지표:** 추종 RMSE, 평균 u², 안전 종료 수. 목표별 분리 평가.
<!-- /exercise-guidance:reinforcement-learning/week-09/case-studies -->


## Gymnasium에서 로봇 Simulator로

<!-- exercise-guidance:reinforcement-learning/week-09/activity-1 -->
### 시작 조건

CartPole-v1의 관측·행동·Reset·시간 제한을 출발점으로 삼습니다.

### 진행 방법

1. 제공된 Rule policy를 실행하고 Episode 하나의 관측·행동을 기록하세요.
2. Isaac Lab 공식 Quickstart에서 Task·Physics backend·Renderer·RL library의 역할을 찾아 대응시키세요.

### 남길 결과

- 환경 인터페이스 대응 표와 Rollout 한 개.
<!-- /exercise-guidance:reinforcement-learning/week-09/activity-1 -->


In [ ]:
"""Small public-environment benchmark; evaluation traces are ordinary numeric arrays."""
import json
from importlib.metadata import version
from pathlib import Path
import numpy as np
import gymnasium as gym

def collect(env_id='CartPole-v1', policy=None, seeds=range(100,103)):
    env=gym.make(env_id); transitions=[]; summary=[]
    for seed in seeds:
        obs,info=env.reset(seed=seed);env.action_space.seed(seed);total=0.;length=0
        while True:
            action=env.action_space.sample() if policy is None else policy(obs)
            nxt,reward,terminated,truncated,info=env.step(action)
            transitions.append((obs.copy(),np.asarray(action).copy(),reward,nxt.copy(),terminated,truncated,seed))
            obs=nxt;total+=reward;length+=1
            if terminated or truncated:break
        summary.append([seed,total,length,int(terminated),int(truncated)])
    env.close()
    columns=list(zip(*transitions))
    data={k:np.asarray(v) for k,v in zip(['observations','actions','rewards','next_observations','terminated','truncated','episode_seed'],columns)}
    return np.asarray(summary),data

def cartpole_rule(obs):return int(obs[2]+.25*obs[3]>0)
def zero_torque(obs):return np.zeros(1,dtype=np.float32)

def save_trace(path, env_id, data, policy_name, seeds):
    # Contains numeric arrays and JSON text only; never requires pickle.
    metadata=dict(env_id=env_id,policy=policy_name,seeds=list(seeds),gymnasium=version('gymnasium'))
    np.savez_compressed(path,**data,metadata_json=np.array(json.dumps(metadata)))
    with np.load(path,allow_pickle=False) as saved:
        assert len(saved['observations'])==len(saved['rewards'])
        print(json.loads(str(saved['metadata_json'])))

def public_model(algorithm='DQN',env_id='CartPole-v1',seed=7,steps=2048,**changes):
    from stable_baselines3 import DQN,PPO,SAC
    options=dict(policy_kwargs=dict(net_arch=[32,32]),seed=seed,verbose=0,device='cpu')
    if algorithm=='DQN':options.update(buffer_size=5000,learning_starts=100,target_update_interval=100,batch_size=32)
    if algorithm=='PPO':options.update(n_steps=128,batch_size=64,n_epochs=3)
    if algorithm=='SAC':options.update(buffer_size=5000,learning_starts=100,batch_size=64)
    options.update(changes)
    env=gym.make(env_id)
    model={'DQN':DQN,'PPO':PPO,'SAC':SAC}[algorithm]('MlpPolicy',env,**options)
    model.learn(total_timesteps=steps)
    env.close()
    return model

print('Gymnasium',version('gymnasium'))
env=gym.make('CartPole-v1')
print('Observation:',env.observation_space,'Action:',env.action_space,'Limit:',env.spec.max_episode_steps)
env.close()
# For visual inspection, create a separate env with render_mode='rgb_array'.
# Rendering requires gymnasium[classic-control]; do not render every training step.

rows,trace=collect(policy=cartpole_rule,seeds=[100]);print(rows)


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-09/activity-1-record -->
**역할:**

**Gymnasium 예:**

**Isaac Lab 설정:**

**내가 바꿀 항목:**

### 확인할 점

- 물리 시뮬레이션과 학습 알고리즘을 같은 구성요소로 혼동하지 않았는가?
- Policy 제어 주기와 물리 시뮬레이션 주기를 구분했는가?
<!-- /exercise-guidance:reinforcement-learning/week-09/activity-1-record -->


## 환경 온보딩: 실행 또는 Transition 분석

<!-- exercise-guidance:reinforcement-learning/week-09/activity-2 -->
### 시작 조건

GPU 환경: 준비된 Linux/Windows NVIDIA 환경에서 공식 Quickstart의 Task 목록→Zero/Random agent→학습→Play 순으로 진행합니다. CPU 환경: 같은 시간에 CartPole Transition·정책 비교와 Task 설정 대응을 완성합니다.

### 진행 방법

1. GPU 환경이 준비되어 있으면 공식 문서의 현재 버전에 맞춰 Cartpole Task를 실행하고 Task ID·backend·seed·실행 명령을 기록하세요.
2. GPU 환경이 없으면 제공된 CPU 코드에서 Random과 Rule의 궤적을 비교하고 종료 원인을 찾으세요.
3. 선택 경로에서 관측·행동·종료 규칙 중 한 항목을 바꾸려면 어느 설정을 수정할지 찾으세요.

### 남길 결과

- GPU: 실행 화면 또는 Rollout과 실행 조건. CPU: 상태 궤적 그림과 설정 대응 표.
<!-- /exercise-guidance:reinforcement-learning/week-09/activity-2 -->


In [ ]:
import matplotlib.pyplot as plt
for name,policy in [('random',None),('rule',cartpole_rule)]:
    rows,trace=collect(policy=policy,seeds=[100]);plt.plot(trace['observations'][:,2],label=name)
plt.xlabel('Control step');plt.ylabel('Pole angle [rad]');plt.legend()


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-09/activity-2-record -->
**경로:**

**Task·환경:**

**실행 조건:**

**확인한 결과:**

**수정할 설정:**

### 확인할 점

- 실행하지 않은 Isaac 결과를 실행 결과처럼 적지 않았는가?
- 설치 문제 해결만으로 끝나지 않고 관측–행동–종료를 추적했는가?
<!-- /exercise-guidance:reinforcement-learning/week-09/activity-2-record -->


## 연구 Task의 최소 구성

<!-- exercise-guidance:reinforcement-learning/week-09/activity-3 -->
### 시작 조건

Manipulation·이동로봇·공정 제어 중 하나를 선택합니다.

### 진행 방법

1. 장면·상태 관측·Action scale·Reward·Reset·Termination을 한 표에 적으세요.
2. Isaac Lab 공식 로봇 튜토리얼에서 해당 설정이 있는 파일·구성요소를 연결하세요.
3. 첫 실험에서는 무엇을 고정하고 어떤 한 항목만 바꿀지 정하세요.

### 남길 결과

- 최소 Task 설계 표와 첫 검증 계획.
<!-- /exercise-guidance:reinforcement-learning/week-09/activity-3 -->


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-09/activity-3-record -->
**구성요소:**

**내 Task 정의:**

**참고 설정 위치:**

**첫 검증:**

### 확인할 점

- 시각화가 되는 것과 학습 문제의 정의가 올바른 것을 구분했는가?
- 처음부터 완전한 로봇 시스템을 재현하려 하지 않고 검증할 핵심을 정했는가?
<!-- /exercise-guidance:reinforcement-learning/week-09/activity-3-record -->


## 공식 예제·데이터

<!-- exercise-guidance:reinforcement-learning/week-09/resources -->
- [Isaac Lab · Quickstart](https://isaac-sim.github.io/IsaacLab/main/source/isaaclab/quickstart.html)
- [Isaac Lab · SO-101 Task 튜토리얼](https://isaac-sim.github.io/IsaacLab/develop/source/isaaclab/tutorial.html)
<!-- /exercise-guidance:reinforcement-learning/week-09/resources -->
